In [1]:
# Specifying the exact file path
file_path = "/home/imran/backup/BBCScraper/data/all.txt"

# opening and reading the file safely
with open(file_path, mode="r", encoding="utf-8") as file_handle:
    raw_text = file_handle.read()
# Checking the results
print(f"File successfully loaded! Total characters: {len(raw_text)}")


File successfully loaded! Total characters: 3541516


In [2]:
import re
def clean_urdu_text(text):
    #1. Define the range of valid Urdu Unicode characters
    urdu_range = r'[\u0600-\u06FF\u0750-\u077F\uFB50-\uFDFF\uFE70-\uFEFF]'
    # 2. Keep only valid Urdu characters and spaces.
    cleaned = re.sub(f"[^{urdu_range[1:-1]}\s]", "", text)
    # 3. Collapse multiple spaces into a single space
    cleaned = re.sub(r'\s+', ' ', cleaned)
    # 4. Strip leading and trailing whitespace from the entire document
    return cleaned.strip()

# test_input = "which essentially means that according to BBC Pakistan is a beautifull country!بی بی سی اردو: پاکستان ایک خوبصورت ملک ہے! (2026)"

# print(clean_urdu_text(test_input))    

In [3]:
def normalize_urdu_characters(text):
    # A simple dictionary replacing Arabic variants with standard Urdu Unicode equivalents
    corrections = {
        "\u0647": "\u06c1",  # Arabic Heh to Urdu Chohti Heh
        "\u064a": "\u06cc",  # Arabic Yeh to Urdu Chohti Yeh
    }
    for bad_char, good_char in corrections.items():
        text = text.replace(bad_char, good_char)
    return text

In [4]:
clean_urdu_text = clean_urdu_text(raw_text)
normalized_urdu_text = normalize_urdu_characters(clean_urdu_text)
print(normalized_urdu_text[:500])

پی ٹی آئی کی ٹیکس ایمنسٹی سکیم ن لیگ کی سکیم سے کیسے مختلف ہے؟ وفاقی کابینہ نے ٹیکس ایمنسٹی سکیم کی باضابطہ منظوری دے دی ہے اور اس کا نفاذ صدارتی آرڈیننس کے ذریعے عمل میں لایا جائے گا۔ حکومت کا کہنا ہے کہ اس سکیم کا مقصد محصولات اکٹھے کرنا نہیں بلکہ بے نامی جائیدادوں کو قانون کے دائرے میں لے کر آنا ہے۔ منگل کو وزیر اعظم عمران خان کی زیر صدارت وفاقی کابینہ کے اجلاس میں اس سکیم کی منظوری دی گئی۔ مشیر خزانہ حفیظ شیخ نے میڈیا کے نمائندوں کو اس ایمنسٹی سکیم کے خدو خال بتاتے ہوئے کہا کہ ان افراد کے عل


In [6]:
#1. Tokenize the cleaned string into a list of words
urdu_words = normalized_urdu_text.split()
# 2. Verify the output using list slicing
print(f"Total tokens (words) in dataset: {len(urdu_words)}")
print("\nFirst 10 tokenized words:")
print(urdu_words[:10])


Total tokens (words) in dataset: 762377

First 10 tokenized words:
['پی', 'ٹی', 'آئی', 'کی', 'ٹیکس', 'ایمنسٹی', 'سکیم', 'ن', 'لیگ', 'کی']


In [7]:
from collections import Counter

# Count how many times each unique word appears
unigram_counts = Counter(urdu_words)

# Let's see your vocabulary size and top words
print(f"Total Unique Words (Vocabulary Size): {len(unigram_counts)}")
print("\nTop 5 most common Urdu words in your dataset:")
print(unigram_counts.most_common(5))

Total Unique Words (Vocabulary Size): 26190

Top 5 most common Urdu words in your dataset:
[('کے', 31972), ('میں', 24922), ('کی', 21459), ('سے', 16283), ('اور', 14247)]


In [8]:
# Create the bigram pairs
# zip(['A', 'B', 'C'], ['B', 'C']) -> [('A', 'B'), ('B', 'C')]
bigram_pairs = list(zip(urdu_words, urdu_words[1:]))

# Count how many times each unique pair appears
bigram_counts = Counter(bigram_pairs)

print(f"Total Unique Bigrams: {len(bigram_counts)}")
print("\nTop 5 most common word pairs:")
print(bigram_counts.most_common(5))

Total Unique Bigrams: 240361

Top 5 most common word pairs:
[(('ہے', 'کہ'), 4274), (('کے', 'لیے'), 3527), (('انھوں', 'نے'), 1916), (('کا', 'کہنا'), 1768), (('کے', 'بعد'), 1750)]


In [9]:
# Initialize our empty probability matrix
bigram_matrix = {}

# Loop through our bigram counts to calculate probabilities
for (word_a, word_b), count in bigram_counts.items():
    # Get total occurrences of Word A from our unigram counts
    total_word_a = unigram_counts[word_a]
    
    # Calculate the probability
    probability = count / total_word_a
    
    # Safely insert into our nested dictionary structure
    if word_a not in bigram_matrix:
        bigram_matrix[word_a] = {}
        
    bigram_matrix[word_a][word_b] = probability

print("\nMatrix successfully built!")


Matrix successfully built!


In [10]:
test_word = "وزیر"  # Replace with a word you know exists in your text

if test_word in bigram_matrix:
    print(f"\nWords most likely to follow '{test_word}':")
    # Sort the following words by their probability in descending order
    sorted_followers = sorted(bigram_matrix[test_word].items(), key=lambda x: x[1], reverse=True)
    print(sorted_followers[:3]) # Print top 3 target words
else:
    print(f"'{test_word}' not found in the matrix dataset.")


Words most likely to follow 'وزیر':
[('اعظم', 0.43243243243243246), ('خارجہ', 0.13513513513513514), ('اعلیٰ', 0.06177606177606178)]
